In [1]:
# !pip install langchain

In [2]:
# from langchain import PromptTemplate
# print("LangChain is installed and working!")

LangChain is installed and working!


In [3]:
# !pip install -U langchain-community

In [4]:
# !pip show langchain-community

## PDF data

In [1]:
# from langchain.document_loaders import PyPDFLoader

# pdf_loader2 = PyPDFLoader(file_path="OPEN SOURCE SOFTWARE GUIDELINES.pdf")
# data2 = pdf_loader2.load()

# for doc2 in data2:
#     print(doc.page_content)

## Web data

#### Using WebBaseLoader

Web Source 1

In [8]:
from langchain.document_loaders import WebBaseLoader
web_loader = WebBaseLoader("https://licensespring.com/blog/glossary/open-source-software/")
web_data = web_loader.load()
for doc in web_data:
    print(f"Source: {doc.metadata['source']}")
    print(doc.page_content)

USER_AGENT environment variable not set, consider setting it to identify your requests.


Source: https://licensespring.com/blog/glossary/open-source-software/
Open-Source Software Development: An In-Depth Guide by LicenseSpring<iframe src="https://www.googletagmanager.com/ns.html?id=GTM-58MJ95T" height="0" width="0" style="display: none; visibility: hidden" aria-hidden="true"></iframe>Software LicensingSoftware LicensingFloating LicensesMetered LicensingUser-based licensingNode lockingAPIGeneral APIC++.NETJavaView AllPricingIndustriesEnterprise & IndustryEducationGaming & VRISVs and Start-upsDocsBlogLog inVendor PlatformEnd-User PlatformOpen trial account* No credit card requiredHome >  Blog >  Glossary >  Open-Source Software Development: An In-Depth Guide by LicenseSpringThe Ultimate Guide to Open-Source Software Development by LicenseSpringPublished on: May 29, 2024Last updated: July 5, 2024by Kyle BrandonTable of Contents:IntroductionOpen-source software, as a development style, has changed how organizations handle licensing. By encouraging creativity, teamwork, and hi

Web source 2

#### Using RecursiveUrlLoader

In [4]:
!pip install -qU langchain-community beautifulsoup4

In [3]:
from langchain_community.document_loaders import RecursiveUrlLoader

loader = RecursiveUrlLoader(
    "https://licensespring.com/blog/glossary/open-source-software/",
    # max_depth=2,
    # use_async=False,
    # extractor=None,
    # metadata_extractor=None,
    # exclude_dirs=(),
    # timeout=10,
    # check_response_status=True,
    # continue_on_failure=True,
    # prevent_outside=True,
    # base_url=None,
    # ...
)

In [4]:
docs = loader.load()
docs[0].metadata

{'source': 'https://licensespring.com/blog/glossary/open-source-software/',
 'content_type': 'text/html',
 'title': 'Open-Source Software Development: An In-Depth Guide by LicenseSpring',
 'description': 'Discover key aspects of open-source software: licensing, monetization, compliance and popular projects, and see how it drives innovation and collaboration.',
 'language': 'en'}

In [5]:
print(docs[0].page_content[:300])

<!DOCTYPE html><html lang="en"><head><meta charSet="utf-8"/><meta http-equiv="x-ua-compatible" content="ie=edge"/><meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no"/><meta data-react-helmet="true" name="twitter:description" content="Discover key aspects of open-sou


##### Adding extractor: To parse HTML into a more human/LLM-friendly format

In [6]:
import re

from bs4 import BeautifulSoup


def bs4_extractor(html: str) -> str: 
    soup = BeautifulSoup(html, "lxml") #parse the html string using xml parser
    return re.sub(r"\n\n+", "\n\n", soup.text).strip() #extrating text and removing html tags + handling lines


loader = RecursiveUrlLoader("https://licensespring.com/blog/glossary/open-source-software/", extractor=bs4_extractor)
docs = loader.load()
print(docs[0].page_content[:2000])

Open-Source Software Development: An In-Depth Guide by LicenseSpring<iframe src="https://www.googletagmanager.com/ns.html?id=GTM-58MJ95T" height="0" width="0" style="display: none; visibility: hidden" aria-hidden="true"></iframe>Software LicensingSoftware LicensingFloating LicensesMetered LicensingUser-based licensingNode lockingAPIGeneral APIC++.NETJavaView AllPricingIndustriesEnterprise & IndustryEducationGaming & VRISVs and Start-upsDocsBlogLog inVendor PlatformEnd-User PlatformOpen trial account* No credit card requiredHome >  Blog >  Glossary >  Open-Source Software Development: An In-Depth Guide by LicenseSpringThe Ultimate Guide to Open-Source Software Development by LicenseSpringPublished on: May 29, 2024Last updated: July 5, 2024by Kyle BrandonTable of Contents:IntroductionOpen-source software, as a development style, has changed how organizations handle licensing. By encouraging creativity, teamwork, and higher productivity across different industries, it has become a popular

In [7]:
import re

from bs4 import BeautifulSoup

from langchain.utils.html import (PREFIXES_TO_IGNORE_REGEX,
                                  SUFFIXES_TO_IGNORE_REGEX)

# from config import *
# import logging
# import sys
# logging.basicConfig(stream=sys.stdout, level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


#This function is used to process and clean the HTML content fetched from each URL
def bs4_extractor(html: str) -> str: 
    soup = BeautifulSoup(html, "lxml") #parse the html string using xml parser
    return re.sub(r"\n\n+", "\n\n", soup.text).strip() #extrating text and removing html tags + handling lines


loader = RecursiveUrlLoader(
    "https://licensespring.com/blog/glossary/open-source-software/",
    max_depth=4, #Limits the recursion depth to 4 levels, This prevents infinite recursion and helps control resource usage.
    prevent_outside=True, #If set to True, it won't follow links to external domains (e.g., links to example.com from licensespring.com).
    # use_async=True,  #fetching multiple pages simultaneously, improving efficiency.
    timeout=600,  #Specifies the maximum time (in seconds) the loader will wait for a page to respond before timing out.
    extractor=bs4_extractor,
    #filter out unwanted URL prefixes (e.g., mailto:, tel:, or other non-HTTP protocols) and suffixes (e.g., .png, .pdf, .zip)
    link_regex=(
            f"href=[\"']{PREFIXES_TO_IGNORE_REGEX}((?:{SUFFIXES_TO_IGNORE_REGEX}.)*?)" #Matches links with href attributes in HTML
            r"(?:[\#'\"]|\/[\#'\"])" #Ensures the matched link ends with #, ', " or /
)
)
documents = loader.load()

# logging.info("index creating with `%d` documents", len(docs))

print(documents[0].page_content[:5000])
# num_characters = len(documents[0].page_content)
# print(f"Total number of characters in the document: {num_characters}")

Open-Source Software Development: An In-Depth Guide by LicenseSpring<iframe src="https://www.googletagmanager.com/ns.html?id=GTM-58MJ95T" height="0" width="0" style="display: none; visibility: hidden" aria-hidden="true"></iframe>Software LicensingSoftware LicensingFloating LicensesMetered LicensingUser-based licensingNode lockingAPIGeneral APIC++.NETJavaView AllPricingIndustriesEnterprise & IndustryEducationGaming & VRISVs and Start-upsDocsBlogLog inVendor PlatformEnd-User PlatformOpen trial account* No credit card requiredHome >  Blog >  Glossary >  Open-Source Software Development: An In-Depth Guide by LicenseSpringThe Ultimate Guide to Open-Source Software Development by LicenseSpringPublished on: May 29, 2024Last updated: July 5, 2024by Kyle BrandonTable of Contents:IntroductionOpen-source software, as a development style, has changed how organizations handle licensing. By encouraging creativity, teamwork, and higher productivity across different industries, it has become a popular

## Split documents

To handle lengthy text efficiently, the Langchain text splitter divides text into smaller, semantically meaningful units and combines them into larger chunks with defined size and overlap. Here, I used the RecursiveCharacterTextSplitter to process scraped documents into manageable chunks while preserving context continuity.

`chunk_size` and `chunk_overlap` effects to the prompt size

execeed promt size causes error `prompt size exceeds the context window size and cannot be processed`

In [8]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

#chunk size: maximum number of characters each chunk can contain
#chunk oberlap: the number of overlapping characters between consecutive chunks, ensure context continuity between chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200) #Splits each document into chunks
documents = text_splitter.split_documents(documents)
print(documents)

[Document(metadata={'source': 'https://licensespring.com/blog/glossary/open-source-software/', 'content_type': 'text/html', 'title': 'Open-Source Software Development: An In-Depth Guide by LicenseSpring', 'description': 'Discover key aspects of open-source software: licensing, monetization, compliance and popular projects, and see how it drives innovation and collaboration.', 'language': 'en'}, page_content='Open-Source Software Development: An In-Depth Guide by LicenseSpring<iframe src="https://www.googletagmanager.com/ns.html?id=GTM-58MJ95T" height="0" width="0" style="display: none; visibility: hidden" aria-hidden="true"></iframe>Software LicensingSoftware LicensingFloating LicensesMetered LicensingUser-based licensingNode lockingAPIGeneral APIC++.NETJavaView AllPricingIndustriesEnterprise & IndustryEducationGaming & VRISVs and Start-upsDocsBlogLog inVendor PlatformEnd-User PlatformOpen trial account* No credit card requiredHome\xa0>\xa0 Blog\xa0>\xa0 Glossary\xa0>\xa0 Open-Source S

##  Create Vector Embedding

After splitting the text, it is converted into vector embeddings using machine learning models like HuggingFace's all-MiniLM-L6-v2. These high-dimensional vectors capture semantic meanings and enable efficient operations such as grouping, searching, and measuring sentence similarity based on semantic closeness, surpassing traditional keyword-based methods.

In [70]:
!pip install sentence-transformers

In [72]:
!pip install chromadb

  Using cached PyPika-0.48.9.tar.gz (67 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached websocket_client-1.8.0-py3-none-any.whl.metadata (8.0 kB)
  Using cached coloredlogs-15.0.1-py2.py3-none-any.whl.metadata (12 kB)
  Using cached monotonic-1.6-py2.py3-none-any.whl.metadata (1.5 kB)
  Using cached backoff-2.2.1-py3-none-any.whl.metadata (14 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached humanfriendly-10.0-py2.py3-none-any.whl.metadata (9.2 kB)
   ---------------------------------------- 0.0/628.3 kB ? eta -:--:--
   ---------------------------------------- 628.3/628.3 kB 4.7 MB/s 

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.1.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.1 which is incompatible.
mediapipe 0.10.11 requires protobuf<4,>=3.11, but you have protobuf 5.29.1 which is incompatible.
paddlepaddle 2.6.1 requires protobuf<=3.20.2,>=3.1.0; platform_system == "Windows", but you have protobuf 5.29.1 which is incompatible.
tensorflow-intel 2.13.0 requires numpy<=1.24.3,>=1.22, but you have numpy 1.24.4 which is incompatible.
tensorflow-intel 2.13.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 5.29.1 which is incompatible.
tensorflow-intel 2.13.0 requires typing-extensions<4.6.0,>=3.6.6, but you have typing-e

In [73]:
!pip install pymongo

  Using cached dnspython-2.6.1-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/727.4 kB ? eta -:--:--
   ---------------------------------------- 727.4/727.4 kB 4.2 MB/s eta 0:00:00
Using cached dnspython-2.6.1-py3-none-any.whl (307 kB)


In [74]:
!pip install python-dotenv

#### configuration

In [16]:
import os

# Set environment variables directly in the notebook
os.environ['INIT_INDEX'] = 'true'
os.environ['INDEX_PERSIST_DIRECTORY'] = './data1/chromadb'
os.environ['TARGET_URL'] = 'https://open5gs.org/open5gs/docs/'
os.environ['HTTP_PORT'] = '7654'
os.environ['MONGO_HOST'] = 'localhost'
os.environ['MONGO_PORT'] = '27017'
os.environ['MONGO_USER'] = 'testuser'
os.environ['MONGO_PASS'] = 'testpass'


In [17]:

# define init index
INIT_INDEX = os.getenv('INIT_INDEX', 'false').lower() == 'true'

# vector index persist directory
INDEX_PERSIST_DIRECTORY = os.getenv('INDEX_PERSIST_DIRECTORY', "./data1/chromadb")

# target url to scrape
TARGET_URL =  os.getenv('TARGET_URL', "https://open5gs.org/open5gs/docs/")

# http api port
HTTP_PORT = os.getenv('HTTP_PORT', 7654)

# mongodb config host, username, password
MONGO_HOST = os.getenv('MONGO_HOST', 'localhost')
MONGO_PORT = os.getenv('MONGO_PORT', 27017)
MONGO_USER = os.getenv('MONGO_USER', 'testuser')
MONGO_PASS = os.getenv('MONGO_PASS', 'testpass')

In [18]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma

# Initialize embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


## Store Vector Embedding in Chroma

Chroma (ChromaDB) is an open-source vector database that stores embeddings and their metadata, enabling efficient semantic search by processing text-based data semantically, unlike traditional databases. It enhances the system's ability to quickly retrieve and compare relevant information, improving the accuracy of responses to user queries.

In [19]:
# Create the vector database from documents
vectordb = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=INDEX_PERSIST_DIRECTORY
)

In [20]:
# Check if the vector store contains any documents
print(f"Number of documents in the vector store: {len(vectordb)}")

Number of documents in the vector store: 32


In [86]:
!pip install -U langchain-chroma

In [21]:
from langchain_chroma import Chroma
# load index
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectordb = Chroma(persist_directory=INDEX_PERSIST_DIRECTORY,embedding_function=embeddings)

## User Ask Question

The system offers an API that allows users to ask questions related to Open5GS documentation, with user sessions identified by a user_id for tracking. The API is designed for ease of use, enabling intuitive interactions, and in real-world scenarios, user identification could be managed via an Authorization header.

## Create Vector Embedding of Question

When a user submits a question through the API, the system converts it into a vector embedding, which is automatically generated by the ConversationalRetrievalChain, enabling semantic search of relevant documents in the vector database.

In [24]:
from langchain_community.llms import Ollama
from langchain.llms import OpenAI
# llama2 llm which runs with ollama
# ollama expose an api for the llam in `localhost:11434`
llm = Ollama(
    model="llama3.2",
    base_url="http://localhost:11434",
    verbose=True,
)

In [23]:
from langchain.chains import ConversationalRetrievalChain

# global conversation
# conversation = None

# create conversation
conversation = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=vectordb.as_retriever(),
    return_source_documents=True,
    verbose=True,
)

chat_history = []
response = conversation({"question": "Give me the definition of OSS?", "chat_history": chat_history})
answer = response['answer']
print(answer)

C:\Users\Bouchra HP\AppData\Local\Temp\ipykernel_22324\2257857199.py:15: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  response = conversation({"question": "Give me the definition of OSS?", "chat_history": chat_history})
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Prompt after formatting:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

software, as a development style, has changed how organizations handle licensing. By encouraging creativity, teamwork, and higher productivity across different industries, it has become a popular choice for companies. Understanding open-source software is crucial for developers, businesses, and technology enthusiasts because of its growing importance in the market.This article provides an overview of open-source solutions, covering key topics such as an introduction to open-source, ways to monetize OSS, and compliance considerations.What is Open-Source Software?As the name implies, open-source software (OSS) refers to any software project that makes their source code publicly available to view, modify, improve, and often redistribute. These usage rights are governed by the license granted by the auth